# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (AI Referral Opportunity) is best framed as a scoring/ranking task, not classification. Since only about 6.4% of pages have any AI referral sessions, treating this as a binary classification problem would mean training on very few positive examples — the model would look accurate but actually be unreliable. Instead, the goal is to score and rank pages by how likely they are to represent an "AI opportunity," based on patterns in content type, intent, word count, and existing AI traffic — giving a content team an ordered list to review, not a hard yes/no prediction.

In [ ]:
import pandas as pd
import numpy as np

# Verify target distribution showing class imbalance and justification for ranking/scoring
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
ai_positive_pct = (df["ai_sessions_90d"] > 0).mean() * 100
print(f"Pages with AI referral traffic: {ai_positive_pct:.2f}%")
print("Task Choice: Scoring / Ranking (Pointwise ordering for editorial review queue)")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My proxy target would be `ai_traffic_pct` — the percentage of a page's sessions that come from AI referrals — used as a continuous score rather than a strict label. Alternatively, a simplified proxy could be a binary flag: `has_ai_traffic = ai_sessions_90d > 0`. This label comes from an observed outcome (real session data), not a manually defined rule. I'll lean toward using `ai_traffic_pct` as a ranking score rather than a strict classification label, to avoid the unreliability of predicting a rare binary event.

In [ ]:
# Code check: Verify observed label target statistics
print("Observed Outcome Target Summary:")
print(df[["ai_sessions_90d", "ai_traffic_pct"]].describe().T[['mean', 'std', 'min', '50%', 'max']])

## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is Precision@K (e.g., Precision@20 or Precision@50) — of the top K pages my ranking surfaces as "AI opportunities," how many actually have meaningful AI traffic? This fits because the real-world use case is a content team reviewing a limited number of pages, not classifying every page — so what matters is whether the top of the list is genuinely good, not overall accuracy across all 30,000 pages.

In [ ]:
# Demonstration: Calculating Precision@K on top priority candidates
def precision_at_k(df, score_col, target_col, k=50):
    top_k = df.sort_values(by=score_col, ascending=False).head(k)
    return (top_k[target_col] > 0).mean()

# Baseline Precision@50 using raw word count as naive heuristic
baseline_p50 = precision_at_k(df, score_col="word_count", target_col="ai_sessions_90d", k=50)
print(f"Baseline Heuristic Precision@50: {baseline_p50:.2f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one row = one content page (identified by `content_id`). The dataframe below shows a real slice of the data for the AI Referral Opportunity lane: content characteristics (`content_type`, `main_intent`, `word_count`) alongside the AI traffic signals (`ai_sessions_90d`, `ai_traffic_pct`). As expected, most pages show zero AI sessions — confirming how sparse this signal is.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

ai_slice = df[[
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "ai_sessions_90d", "ai_traffic_pct"
]]

print("One row = one content page")
ai_slice.head(10)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag pages with `word_count > 3000` and `main_intent = informational`" is too rigid — AI referral opportunity likely depends on a combination of factors (content type, intent, depth, and existing traffic patterns) interacting in ways that aren't obvious from a single threshold. ML-based scoring can find these patterns from the data itself and rank pages by a combined opportunity score, adapting as more signal becomes available — something a fixed rule can't do without constant manual retuning.

In [ ]:
# Demonstration: Evaluating failure mode of a fixed heuristic rule
rule_matches = df[(df["word_count"] > 3000) & (df["main_intent"] == "informational")]
rule_precision = (rule_matches["ai_sessions_90d"] > 0).mean()

print(f"Total pages matched by static rule: {len(rule_matches)}")
print(f"Precision of static rule: {rule_precision:.4f}")
print("Conclusion: Fixed rule captures too much noise and fails to prioritize high-value candidates.")

## Self-check

Before you submit, confirm each line honestly:

- [ Confirm ] Every section above is filled — markdown thinking AND the code that backs it
- [ Confirm ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Confirm ] No client names, URLs, or private queries anywhere
- [ Confirm ] My claims use careful words: observed, measured, directional, decision-support
- [ Confirm ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.